# Qasper Dataset Loader

Notebook này chạy độc lập trên Kaggle/Colab. Không cần clone GitHub, không cần import code từ repo. Mục tiêu hiện tại: tải và kiểm tra bộ dữ liệu Qasper trước khi xây baseline RAG.

In [ ]:
!pip install -q datasets pyarrow

In [ ]:
from datasets import load_dataset

QASPER_REVISION = "cc58ffb39db7ff6ce1951e28e029996bf499304e"
QASPER_BASE_URL = f"https://huggingface.co/datasets/allenai/qasper/resolve/{QASPER_REVISION}/qasper"
QASPER_PARQUET_FILES = {
    "train": f"{QASPER_BASE_URL}/qasper-train.parquet",
    "validation": f"{QASPER_BASE_URL}/qasper-validation.parquet",
    "test": f"{QASPER_BASE_URL}/qasper-test.parquet",
}

ds = load_dataset("parquet", data_files=QASPER_PARQUET_FILES)
ds

In [ ]:
print(ds.keys())
print(ds["train"])
print(ds["validation"])
print(ds["test"])

In [ ]:
sample = ds["validation"][0]

print("ID:", sample["id"])
print("Title:", sample["title"])
print("Abstract preview:", sample["abstract"][:500])
print("Number of QA pairs:", len(sample["qas"]["question"]))
print("Full text sections:", len(sample["full_text"]["section_name"]))

In [ ]:
def iter_answer_records(answers):
    if isinstance(answers, list):
        return answers
    if not isinstance(answers, dict):
        return []

    answer_values = answers.get("answer", [])
    annotation_ids = answers.get("annotation_id", [])
    worker_ids = answers.get("worker_id", [])

    if isinstance(answer_values, dict):
        answer_values = [answer_values]

    records = []
    for idx, answer_value in enumerate(answer_values):
        row = {"answer": answer_value}
        if isinstance(annotation_ids, list) and idx < len(annotation_ids):
            row["annotation_id"] = annotation_ids[idx]
        if isinstance(worker_ids, list) and idx < len(worker_ids):
            row["worker_id"] = worker_ids[idx]
        records.append(row)
    return records


def show_qa(record, index=0):
    question = record["qas"]["question"][index]
    answers = iter_answer_records(record["qas"]["answers"][index])

    print("Question:", question)
    print("Answers:")
    for answer in answers:
        print(answer)

show_qa(sample, 0)

In [ ]:
class QasperRecordView:
    def __init__(self, record):
        self.record = record

    @property
    def title(self):
        return self.record["title"]

    def document_text(self):
        parts = [self.record.get("abstract", "")]
        full_text = self.record.get("full_text", {})
        sections = full_text.get("section_name", [])
        paragraphs_by_section = full_text.get("paragraphs", [])

        for section, paragraphs in zip(sections, paragraphs_by_section):
            parts.append(f"\n## {section}\n")
            parts.extend(str(paragraph) for paragraph in paragraphs)
        return "\n".join(parts)

    def qa_pairs(self):
        qas = self.record["qas"]
        rows = []
        for question_id, question, answers in zip(
            qas["question_id"], qas["question"], qas["answers"]
        ):
            rows.append({
                "question_id": question_id,
                "question": question,
                "answers": iter_answer_records(answers),
            })
        return rows

view = QasperRecordView(sample)
print(view.title)
print(view.document_text()[:1000])
print("QA pairs:", len(view.qa_pairs()))

In [ ]:
import json
from pathlib import Path

output_path = Path("qasper_validation_preview.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for record in ds["validation"].select(range(5)):
        view = QasperRecordView(record)
        row = {
            "id": record["id"],
            "title": record["title"],
            "document_preview": view.document_text()[:2000],
            "qa_pairs": view.qa_pairs(),
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", output_path)

## Base RAG Training Runner

Phần dưới dùng code `.py` trong repo để chạy baseline RAG. Trên Kaggle/Colab, hãy upload hoặc clone repo vào working directory trước khi chạy các cell này.

In [ ]:
!pip install -q sentence-transformers transformers torch tqdm numpy

In [ ]:
import os
import sys
from pathlib import Path

repo_candidates = [
    Path.cwd(),
    Path('/kaggle/working/long-context-slm-rag'),
    Path('/content/long-context-slm-rag'),
]

repo_dir = next((path for path in repo_candidates if (path / 'src' / 'qasper_base_rag').exists()), None)
if repo_dir is None:
    raise FileNotFoundError('Upload or clone the repo so src/qasper_base_rag exists in the notebook environment.')

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print('Repo:', repo_dir)

In [ ]:
!python -m src.qasper_base_rag.evaluate \
  --split validation \
  --limit 5 \
  --top-k 5 \
  --output-predictions outputs/base_rag_qasper_predictions.jsonl \
  --output-summary outputs/base_rag_qasper_summary.json

In [ ]:
import json

with open('outputs/base_rag_qasper_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

summary